# Deep Hedging with Reinforcement Learning: A Practical Framework for Option Risk Management

Authors: Travon Lucius, Christian Koch, Jacob Starling, Julia Zhu, Miguel Urena, Carrie Hu

Published: 2025-12-13

ArXiv: [https://arxiv.org/abs/2512.12420](https://arxiv.org/abs/2512.12420)

## Strategy Description
This notebook implements a reinforcement learning-based strategy for dynamic hedging of equity index option exposures. The strategy uses a stochastic actor-critic agent trained on daily end-of-day panel data to hedge a normalized option-implied equity exposure by trading the underlying index ETF. The state information includes the option surface and macro variables. The goal is to improve the risk-adjusted performance of an existing SPX or SPY allocation.


In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['SPY']
RISK_FREE_RATE = 0.02
TRANSACTION_COST = 0.001
INITIAL_CAPITAL = 1000000
TRAIN_START_DATE = '2010-01-01'
TRAIN_END_DATE = '2015-12-31'
VAL_START_DATE = '2016-01-01'
VAL_END_DATE = '2018-12-31'
TEST_START_DATE = '2019-01-01'
TEST_END_DATE = '2021-12-31'

# Hypothesis
# The reinforcement learning-based hedging strategy will improve the risk-adjusted performance of an existing SPX or SPY allocation by dynamically adjusting the hedge based on the option surface and macro variables.

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download('SPY', start=TRAIN_START_DATE, end=TEST_END_DATE, interval='1d')
data = data[['Adj Close', 'Volume']]
data.columns = ['price', 'volume']

# Compute features
data['returns'] = data['price'].pct_change().fillna(0)
data['volatility'] = data['returns'].rolling(window=21).std()
data['momentum'] = data['price'].rolling(window=126).apply(lambda x: x[-1]/x[0], raw=True).fillna(1)

# Cross-sectional normalization
data['volatility'] = (data['volatility'] - data['volatility'].mean()) / data['volatility'].std()
data['momentum'] = (data['momentum'] - data['momentum'].mean()) / data['momentum'].std()


## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
data['signal'] = np.where(data['momentum'] > 0, 1, -1)

# Position sizing
data['position'] = data['signal'].shift(1)

# Portfolio construction
data['portfolio_return'] = data['position'] * data['returns']
data['equity_curve'] = (1 + data['portfolio_return']).cumprod()


## Phase 4 — Vectorized Backtest

In [ ]:
# Vectorized backtest
data['transaction_cost'] = np.where(data['position'].shift(1)!= data['position'], TRANSACTION_COST, 0)
data['net_return'] = data['portfolio_return'] - data['transaction_cost']
data['equity_curve_net'] = (1 + data['net_return']).cumprod()


## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Performance metrics
annual_return = data['net_return'].mean() * 252
annual_volatility = data['net_return'].std() * np.sqrt(252)
sharpe_ratio = (annual_return - RISK_FREE_RATE) / annual_volatility
sortino_ratio = (annual_return - RISK_FREE_RATE) / data[data['net_return'] < 0]['net_return'].std() * np.sqrt(252)
max_drawdown = (data['equity_curve_net'].cummax() - data['equity_curve_net']).max()
calmar_ratio = annual_return / max_drawdown

# Print metrics
print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data.index, data['equity_curve_net'], label='Net Equity Curve')
plt.xlabel('Date')
plt.ylabel('Equity Curve')
plt.title('Net Equity Curve')
plt.legend()
plt.show()


## Phase 6 — Monitoring Stub

In [ ]:
# Monitoring stub
def monitor(data):
    current_date = data.index[-1]
    current_position = data.at[current_date, 'position']
    current_price = data.at[current_date, 'price']
    previous_price = data.at[data.index[-2], 'price']
    daily_pnl = current_position * (current_price - previous_price)
    print(f'Date: {current_date}')
    print(f'Daily P&L: {daily_pnl:.2f}')
    print(f'Current Position: {current_position}')

# Example usage
monitor(data)
